In [1]:
# Fabric Notebook: NB_POI_CHANGE_STAGING_SETUP
# =============================================================================
# Deployment/setup notebook for POI Change + PostgreSQL sync staging.
#
# V5 updates:
#   - Auto-migrates new Gold business columns into Shared/COPY staging.
#   - Validates business-column datatypes after migration.
#   - Prevents schema evolution (e.g. is_enrichable) from breaking deployment setup.
#   - Uses the ACTUAL brz_watermark schema:
#       watermark_id, table_name, watermark_column, watermark_value,
#       update_time, status, flow_name, watermark_sequence,
#       last_run_id, error_message
#   - Includes poi_review in Shared STG / COPY STG.
#   - Entity-specific sync key:
#       poi_master             -> poi_id
#       poi_detail             -> poi_id
#       poi_enrichment_display -> poi_id
#       poi_review             -> review_id
#
# Run for deployment/schema changes, NOT as a periodic runtime notebook.
# =============================================================================

import json
from datetime import datetime, timezone

BRONZE_DB = "sgr_visitvn_dev.lh_vv_bronze.dbo"
GOLD_DB = "sgr_visitvn_dev.lh_vv_gold.dbo"

WATERMARK_TBL = f"{BRONZE_DB}.brz_watermark"
PIPELINE_RUN_TBL = f"{BRONZE_DB}.brz_pipeline_run"
PROCESSED_TBL = f"{BRONZE_DB}.poi_change_event_processed"
ENTITY_VERSION_TBL = f"{BRONZE_DB}.poi_change_entity_version"

NOTEBOOK_NAME = "NB_POI_CHANGE_STAGING_SETUP_V5"

ENTITY_CONFIG = {
    "poi_master": {
        "target": f"{GOLD_DB}.poi_master",
        "shared_staging": f"{GOLD_DB}.stg_poi_master_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_master",
        "sync_key": "poi_id",
    },
    "poi_detail": {
        "target": f"{GOLD_DB}.poi_detail",
        "shared_staging": f"{GOLD_DB}.stg_poi_detail_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_detail",
        "sync_key": "poi_id",
    },
    "poi_enrichment_display": {
        "target": f"{GOLD_DB}.poi_enrichment_display",
        "shared_staging": f"{GOLD_DB}.stg_poi_enrichment_display_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_enrichment_display",
        "sync_key": "poi_id",
    },
    "poi_review": {
        "target": f"{GOLD_DB}.poi_review",
        "shared_staging": f"{GOLD_DB}.stg_poi_review_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_review",
        "sync_key": "review_id",
    },
}

SHARED_META_TYPES = {
    "change_token": "STRING",
    "producer_flow": "STRING",
    "producer_run_id": "STRING",
    "source_event_id": "STRING",
    "source_version": "BIGINT",
    "staged_at": "TIMESTAMP",
}

COPY_META_TYPES = {
    "change_token": "STRING",
    "sync_run_id": "STRING",
    "snapshot_at": "TIMESTAMP",
}

DEPRECATED_SHARED_META = {
    "sync_id", "batch_id", "sync_status", "synced_at", "sync_error"
}

ALL_TRANSPORT_META = (
    set(SHARED_META_TYPES)
    | set(COPY_META_TYPES)
    | DEPRECATED_SHARED_META
)

# Actual schema confirmed from Fabric.
REQUIRED_WATERMARK_COLUMNS = {
    "watermark_id",
    "table_name",
    "watermark_column",
    "watermark_value",
    "update_time",
    "status",
    "flow_name",
    "watermark_sequence",
    "last_run_id",
    "error_message",
}

REQUIRED_PIPELINE_RUN_COLUMNS = {
    "run_id", "flow_name", "status", "started_at"
}


def qcol(name):
    return f"`{name.replace('`', '``')}`"


def table_exists(name):
    try:
        _ = spark.table(name).schema
        return True
    except Exception:
        return False


def columns(name):
    return list(spark.table(name).columns)


def column_set(name):
    return set(columns(name))


def row_count(name):
    return int(spark.sql(f"SELECT COUNT(*) c FROM {name}").first()["c"])


def null_key_count(name, key):
    return int(spark.sql(f"""
        SELECT COUNT(*) c FROM {name}
        WHERE {qcol(key)} IS NULL
    """).first()["c"])


def duplicate_key_groups(name, key):
    return int(spark.sql(f"""
        SELECT COUNT(*) c
        FROM (
            SELECT {qcol(key)}
            FROM {name}
            WHERE {qcol(key)} IS NOT NULL
            GROUP BY {qcol(key)}
            HAVING COUNT(*) > 1
        ) x
    """).first()["c"])


def blank_token_count(name):
    if "change_token" not in column_set(name):
        return 0
    return int(spark.sql(f"""
        SELECT COUNT(*) c FROM {name}
        WHERE change_token IS NULL OR trim(change_token) = ''
    """).first()["c"])


def validate_columns(name, required, label):
    missing = sorted(set(required) - column_set(name))
    if missing:
        raise RuntimeError(
            f"{label}_SCHEMA_MISMATCH table={name} missing={missing}"
        )


def add_missing_columns(name, defs):
    current = column_set(name)
    missing = {k: v for k, v in defs.items() if k not in current}
    if not missing:
        return []
    ddl = ", ".join(f"{qcol(k)} {v}" for k, v in missing.items())
    spark.sql(f"ALTER TABLE {name} ADD COLUMNS ({ddl})")
    return list(missing)


def target_business_types(target):
    """Return target business column -> Spark SQL datatype."""
    return {
        f.name: f.dataType.simpleString()
        for f in spark.table(target).schema.fields
    }


def add_missing_business_columns(staging, target):
    """
    Self-heal staging schema when Gold target gets new business columns.
    Existing staging rows receive NULL for newly added columns.
    No existing column is dropped or overwritten.
    """
    target_types = target_business_types(target)
    current = column_set(staging)
    missing = {c: dtype for c, dtype in target_types.items() if c not in current}
    if not missing:
        return []
    ddl = ", ".join(f"{qcol(c)} {dtype}" for c, dtype in missing.items())
    spark.sql(f"ALTER TABLE {staging} ADD COLUMNS ({ddl})")
    return list(missing)


def validate_business_types(staging, target, label):
    """Fail fast if same-name business columns have incompatible datatypes."""
    target_types = {f.name: f.dataType.simpleString() for f in spark.table(target).schema.fields}
    staging_types = {f.name: f.dataType.simpleString() for f in spark.table(staging).schema.fields}
    mismatches = {
        c: {"target": target_types[c], "staging": staging_types[c]}
        for c in target_types
        if c in staging_types and target_types[c] != staging_types[c]
    }
    if mismatches:
        raise RuntimeError(
            f"{label}_TYPE_MISMATCH table={staging} target={target} mismatches={mismatches}"
        )


# =============================================================================
# 1. GOLD TARGET PREFLIGHT
# =============================================================================

print("=" * 100)
print(f"{NOTEBOOK_NAME}: TARGET PREFLIGHT")
print("=" * 100)

for entity, cfg in ENTITY_CONFIG.items():
    target = cfg["target"]
    key = cfg["sync_key"]

    if not table_exists(target):
        raise RuntimeError(
            f"GOLD_TARGET_NOT_FOUND entity={entity} table={target}"
        )

    target_cols = column_set(target)

    if key not in target_cols:
        raise RuntimeError(
            f"GOLD_TARGET_SYNC_KEY_MISSING entity={entity} "
            f"table={target} sync_key={key}"
        )

    collisions = sorted(target_cols & ALL_TRANSPORT_META)
    if collisions:
        raise RuntimeError(
            f"GOLD_TARGET_METADATA_COLLISION entity={entity} "
            f"columns={collisions}"
        )

    print(
        f"GOLD OK entity={entity} sync_key={key} "
        f"business_columns={len(target_cols)}"
    )


# =============================================================================
# 2. SHARED STAGING CREATE / MIGRATE
# =============================================================================

shared_result = {}

for entity, cfg in ENTITY_CONFIG.items():
    target = cfg["target"]
    shared = cfg["shared_staging"]
    key = cfg["sync_key"]
    target_cols = columns(target)

    created = False
    added_business = []
    added_meta = []

    if not table_exists(shared):
        business_expr = ",\n                ".join(
            f"t.{qcol(c)}" for c in target_cols
        )
        meta_expr = ",\n                ".join(
            f"CAST(NULL AS {dtype}) AS {qcol(name)}"
            for name, dtype in SHARED_META_TYPES.items()
        )

        spark.sql(f"""
            CREATE TABLE {shared}
            USING DELTA
            AS
            SELECT
                {business_expr},
                {meta_expr}
            FROM {target} t
            WHERE 1 = 0
        """)
        created = True

    else:
        # V5: auto-migrate business columns added to Gold target.
        # Example: poi_master.is_enrichable. Existing rows get NULL.
        added_business = add_missing_business_columns(shared, target)
        if added_business:
            print(
                f"SHARED AUTO-MIGRATED entity={entity} "
                f"added_business_columns={sorted(added_business)}"
            )

        added_meta = add_missing_columns(shared, SHARED_META_TYPES)
        validate_business_types(shared, target, "SHARED_STAGING")

    validate_columns(
        shared,
        set(target_cols) | set(SHARED_META_TYPES),
        "SHARED_STAGING"
    )

    null_keys = null_key_count(shared, key)
    dup_keys = duplicate_key_groups(shared, key)

    if null_keys:
        raise RuntimeError(
            f"SHARED_NULL_SYNC_KEY entity={entity} "
            f"sync_key={key} count={null_keys}"
        )

    if dup_keys:
        raise RuntimeError(
            f"SHARED_DUPLICATE_SYNC_KEY entity={entity} "
            f"sync_key={key} duplicate_groups={dup_keys}. "
            "Setup will not auto-deduplicate."
        )

    # Legacy rows: backfill a deterministic token once.
    if blank_token_count(shared) > 0:
        shared_cols = column_set(shared)
        source_version_expr = (
            "CAST(source_version AS STRING)"
            if "source_version" in shared_cols else "''"
        )
        staged_at_expr = (
            "CAST(staged_at AS STRING)"
            if "staged_at" in shared_cols else "''"
        )

        spark.sql(f"""
            UPDATE {shared}
            SET change_token = sha2(
                concat_ws(
                    '||',
                    CAST({qcol(key)} AS STRING),
                    coalesce({source_version_expr}, ''),
                    coalesce({staged_at_expr}, ''),
                    'MIGRATED_SHARED_STAGING_V4'
                ),
                256
            )
            WHERE change_token IS NULL OR trim(change_token) = ''
        """)

    spark.sql(f"""
        UPDATE {shared}
        SET producer_flow = 'MIGRATED_LEGACY_STAGING'
        WHERE producer_flow IS NULL
          AND change_token IS NOT NULL
    """)

    spark.sql(f"""
        UPDATE {shared}
        SET staged_at = current_timestamp()
        WHERE staged_at IS NULL
          AND change_token IS NOT NULL
    """)

    remaining_blank = blank_token_count(shared)
    if remaining_blank:
        raise RuntimeError(
            f"SHARED_TOKEN_BACKFILL_FAILED entity={entity} "
            f"remaining={remaining_blank}"
        )

    shared_result[entity] = {
        "table": shared,
        "sync_key": key,
        "created": created,
        "added_business_columns": added_business,
        "added_metadata_columns": added_meta,
        "row_count": row_count(shared),
        "deprecated_columns_present": sorted(
            DEPRECATED_SHARED_META & column_set(shared)
        ),
    }

    print(
        f"SHARED READY entity={entity} sync_key={key} "
        f"rows={shared_result[entity]['row_count']}"
    )


# =============================================================================
# 3. COPY STAGING CREATE / VALIDATE
# =============================================================================

copy_result = {}

for entity, cfg in ENTITY_CONFIG.items():
    target = cfg["target"]
    copy_stg = cfg["copy_staging"]
    key = cfg["sync_key"]
    target_cols = columns(target)

    created = False
    added_business = []
    added_meta = []

    if not table_exists(copy_stg):
        business_expr = ",\n                ".join(
            f"t.{qcol(c)}" for c in target_cols
        )
        meta_expr = ",\n                ".join(
            f"CAST(NULL AS {dtype}) AS {qcol(name)}"
            for name, dtype in COPY_META_TYPES.items()
        )

        spark.sql(f"""
            CREATE TABLE {copy_stg}
            USING DELTA
            AS
            SELECT
                {business_expr},
                {meta_expr}
            FROM {target} t
            WHERE 1 = 0
        """)
        created = True

    else:
        # V5: auto-migrate both target business columns and COPY metadata.
        added_business = add_missing_business_columns(copy_stg, target)
        added_meta = add_missing_columns(copy_stg, COPY_META_TYPES)

        if added_business or added_meta:
            print(
                f"COPY AUTO-MIGRATED entity={entity} "
                f"added_business_columns={sorted(added_business)} "
                f"added_metadata_columns={sorted(added_meta)}"
            )

        validate_business_types(copy_stg, target, "COPY_STAGING")

    validate_columns(
        copy_stg,
        set(target_cols) | set(COPY_META_TYPES),
        "COPY_STAGING"
    )

    null_keys = null_key_count(copy_stg, key)
    dup_keys = duplicate_key_groups(copy_stg, key)

    if null_keys:
        raise RuntimeError(
            f"COPY_NULL_SYNC_KEY entity={entity} "
            f"sync_key={key} count={null_keys}"
        )

    if dup_keys:
        raise RuntimeError(
            f"COPY_DUPLICATE_SYNC_KEY entity={entity} "
            f"sync_key={key} duplicate_groups={dup_keys}"
        )

    copy_result[entity] = {
        "table": copy_stg,
        "sync_key": key,
        "created": created,
        "added_business_columns": added_business,
        "added_metadata_columns": added_meta,
        "row_count": row_count(copy_stg),
    }

    print(
        f"COPY READY entity={entity} sync_key={key} "
        f"rows={copy_result[entity]['row_count']}"
    )


# =============================================================================
# 4. CHANGE PROCESSOR CONTROL TABLES
# =============================================================================

if not table_exists(PROCESSED_TBL):
    spark.sql(f"""
        CREATE TABLE {PROCESSED_TBL} (
            event_id STRING,
            poi_id STRING,
            entity STRING,
            version BIGINT,
            event_type STRING,
            source STRING,
            changes_json STRING,
            status STRING,
            batch_id STRING,
            processed_at TIMESTAMP,
            error_message STRING
        )
        USING DELTA
    """)
else:
    validate_columns(
        PROCESSED_TBL,
        {
            "event_id", "poi_id", "entity", "version",
            "event_type", "source", "changes_json",
            "status", "processed_at", "error_message"
        },
        "PROCESSED_CONTROL"
    )

if not table_exists(ENTITY_VERSION_TBL):
    spark.sql(f"""
        CREATE TABLE {ENTITY_VERSION_TBL} (
            poi_id STRING,
            entity STRING,
            last_version BIGINT,
            last_event_id STRING,
            updated_at TIMESTAMP
        )
        USING DELTA
    """)
else:
    validate_columns(
        ENTITY_VERSION_TBL,
        {
            "poi_id", "entity", "last_version",
            "last_event_id", "updated_at"
        },
        "ENTITY_VERSION_CONTROL"
    )


# =============================================================================
# 5. ACTUAL SHARED CONTROL SCHEMA VALIDATION
# =============================================================================

if not table_exists(WATERMARK_TBL):
    raise RuntimeError(
        f"WATERMARK_TABLE_NOT_FOUND table={WATERMARK_TBL}"
    )

validate_columns(
    WATERMARK_TBL,
    REQUIRED_WATERMARK_COLUMNS,
    "WATERMARK"
)

print(f"WATERMARK OK table={WATERMARK_TBL}")

if not table_exists(PIPELINE_RUN_TBL):
    raise RuntimeError(
        f"PIPELINE_RUN_TABLE_NOT_FOUND table={PIPELINE_RUN_TBL}"
    )

validate_columns(
    PIPELINE_RUN_TBL,
    REQUIRED_PIPELINE_RUN_COLUMNS,
    "PIPELINE_RUN"
)

print(f"PIPELINE RUN OK table={PIPELINE_RUN_TBL}")


# =============================================================================
# 6. FINAL VALIDATION
# =============================================================================

validation = {}

for entity, cfg in ENTITY_CONFIG.items():
    target = cfg["target"]
    shared = cfg["shared_staging"]
    copy_stg = cfg["copy_staging"]
    key = cfg["sync_key"]

    target_cols = set(columns(target))

    validate_columns(
        shared,
        target_cols | set(SHARED_META_TYPES),
        "FINAL_SHARED"
    )
    validate_columns(
        copy_stg,
        target_cols | set(COPY_META_TYPES),
        "FINAL_COPY"
    )

    validate_business_types(shared, target, "FINAL_SHARED")
    validate_business_types(copy_stg, target, "FINAL_COPY")

    shared_null = null_key_count(shared, key)
    shared_dup = duplicate_key_groups(shared, key)
    copy_null = null_key_count(copy_stg, key)
    copy_dup = duplicate_key_groups(copy_stg, key)

    if shared_null or shared_dup or copy_null or copy_dup:
        raise RuntimeError(
            f"FINAL_SYNC_KEY_INVARIANT_FAILED entity={entity} "
            f"sync_key={key} shared_null={shared_null} "
            f"shared_dup={shared_dup} copy_null={copy_null} "
            f"copy_dup={copy_dup}"
        )

    validation[entity] = {
        "sync_key": key,
        "target": target,
        "shared": shared,
        "copy": copy_stg,
        "shared_rows": row_count(shared),
        "copy_rows": row_count(copy_stg),
        "unique_key_invariant": True,
    }


# =============================================================================
# 7. RESULT
# =============================================================================

result = {
    "status": "SUCCESS",
    "notebook": NOTEBOOK_NAME,
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "watermark_contract": {
        "table": WATERMARK_TBL,
        "columns": sorted(REQUIRED_WATERMARK_COLUMNS),
    },
    "entities": {
        entity: {
            "sync_key": cfg["sync_key"],
            "target": cfg["target"],
            "shared_staging": cfg["shared_staging"],
            "copy_staging": cfg["copy_staging"],
        }
        for entity, cfg in ENTITY_CONFIG.items()
    },
    "shared_staging_setup": shared_result,
    "copy_staging_setup": copy_result,
    "final_validation": validation,
}

result_json = json.dumps(result, ensure_ascii=False, default=str)
print("=" * 100)
print(result_json)
print("=" * 100)

try:
    notebookutils.notebook.exit(result_json)
except NameError:
    pass


StatementMeta(, 92511167-7ea3-47c3-ae7a-0c65778a3c89, 3, Finished, Available, Finished, False)

NB_POI_CHANGE_STAGING_SETUP_V5: TARGET PREFLIGHT
GOLD OK entity=poi_master sync_key=poi_id business_columns=31
GOLD OK entity=poi_detail sync_key=poi_id business_columns=18
GOLD OK entity=poi_enrichment_display sync_key=poi_id business_columns=18
GOLD OK entity=poi_review sync_key=review_id business_columns=25
SHARED AUTO-MIGRATED entity=poi_master added_business_columns=['is_enrichable']
SHARED READY entity=poi_master sync_key=poi_id rows=23511
SHARED READY entity=poi_detail sync_key=poi_id rows=23511
SHARED READY entity=poi_enrichment_display sync_key=poi_id rows=23511
SHARED READY entity=poi_review sync_key=review_id rows=107690
COPY AUTO-MIGRATED entity=poi_master added_business_columns=['is_enrichable'] added_metadata_columns=[]
COPY READY entity=poi_master sync_key=poi_id rows=17082
COPY READY entity=poi_detail sync_key=poi_id rows=17082
COPY READY entity=poi_enrichment_display sync_key=poi_id rows=17082
COPY READY entity=poi_review sync_key=review_id rows=75367
WATERMARK OK tabl